# Learning Rate Schedules in PyTorch

This notebook demonstrates and compares six common **learning rate scheduling strategies** available in PyTorch. A well-chosen schedule can dramatically improve convergence speed and final model quality. We first visualize each schedule's learning rate curve, then train a CNN on Fashion-MNIST with different optimizer/scheduler combinations to see their effect on real training dynamics.

## Why does learning rate scheduling matter?

A fixed learning rate faces a fundamental tension:
- **Too large:** training oscillates or diverges near the optimum.
- **Too small:** training converges slowly and may get trapped in sharp local minima.

Learning rate schedules resolve this by starting with a larger rate for fast initial progress and gradually reducing it for fine-grained convergence. Some schedules also include a **warmup** phase that ramps the rate up from a small value, which stabilizes early training when gradients are noisy (e.g., with random initialization or large batches).

## Schedules covered

| Schedule | Formula / Description |
|---|---|
| **StepLR** | Multiply by $\gamma$ every `step_size` epochs: $\alpha_t = \alpha_0 \cdot \gamma^{\lfloor t / S \rfloor}$ |
| **MultiStepLR** | Multiply by $\gamma$ at specified milestone epochs |
| **ExponentialLR** | $\alpha_t = \alpha_0 \cdot \gamma^t$ -- smooth exponential decay every epoch |
| **PolynomialLR** | $\alpha_t = \alpha_0 \cdot \bigl(1 - t/T\bigr)^p$ -- polynomial decay to zero |
| **CosineAnnealingLR** | $\alpha_t = \alpha_{\min} + \frac{1}{2}(\alpha_{\max} - \alpha_{\min})\bigl(1 + \cos(\pi\, t / T)\bigr)$ -- smooth cosine curve |
| **OneCycleLR** | Linear warmup to `max_lr`, then cosine annealing to a very small rate -- the "super-convergence" policy |

### Practical guidance

- **StepLR / MultiStepLR** are the simplest and work well when you know roughly where training plateaus. Common in older vision pipelines.
- **CosineAnnealingLR** is a strong default for most tasks -- it provides a smooth, theoretically motivated decay and is widely used with SGD in modern deep learning.
- **OneCycleLR** (warmup + cosine) often achieves the best results in the fewest epochs ("super-convergence"). It is recommended when training from scratch with SGD or AdamW.
- **ExponentialLR** decays aggressively and is best for short training runs.
- **PolynomialLR** with power $> 1$ gives a slow start and fast finish; it is popular in NLP fine-tuning.

## Visualizing the Six Schedules

Below we create a dummy model and optimizer (initial $\alpha_0 = 0.1$) and step each scheduler for 100 epochs, recording the learning rate at each epoch. The resulting curves show the qualitative shape of each schedule.

In [ ]:
import torch
import matplotlib.pyplot as plt

# Define a simple model and optimizer
model = torch.nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# Number of epochs for simulation
num_epochs = 100

# Define various schedulers
scheduler_multistep = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 60, 90], gamma=0.1)
scheduler_exponential = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)
scheduler_polynomial = torch.optim.lr_scheduler.PolynomialLR(optimizer, total_iters=num_epochs, power=2)
scheduler_step = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# OneCycleLR: linear warmup (30% of epochs) then cosine annealing
scheduler_onecycle = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.1,
    total_steps=num_epochs,
    pct_start=0.3,               # 30% warmup
    anneal_strategy='cos',
    div_factor=25.0,             # initial_lr = max_lr / 25
    final_div_factor=10000.0     # final_lr  = initial_lr / 10000
)

# Collect learning rates for each scheduler
lrs_multistep = []
lrs_exponential = []
lrs_polynomial = []
lrs_step = []
lrs_cosine = []
lrs_onecycle = []

def collect_lrs(scheduler, lrs_list):
    """Run scheduler for num_epochs steps, recording the LR at each epoch."""
    optimizer.param_groups[0]["lr"] = 0.1  # reset initial LR
    for _ in range(num_epochs):
        optimizer.step()
        lrs_list.append(optimizer.param_groups[0]["lr"])
        scheduler.step()

collect_lrs(scheduler_multistep, lrs_multistep)
collect_lrs(scheduler_exponential, lrs_exponential)
collect_lrs(scheduler_polynomial, lrs_polynomial)
collect_lrs(scheduler_step, lrs_step)
collect_lrs(scheduler_cosine, lrs_cosine)
collect_lrs(scheduler_onecycle, lrs_onecycle)

# --- Combined plot ---
plt.figure(figsize=(15, 8))
plt.plot(lrs_multistep, label="MultiStepLR", linewidth=2, linestyle='-')
plt.plot(lrs_exponential, label="ExponentialLR", linewidth=2, linestyle='--')
plt.plot(lrs_polynomial, label="PolynomialLR", linewidth=2, linestyle='-.')
plt.plot(lrs_step, label="StepLR", linewidth=2, linestyle=':')
plt.plot(lrs_cosine, label="CosineAnnealingLR", linewidth=2, linestyle='-.')
plt.plot(lrs_onecycle, label="OneCycleLR (Warmup + Cosine)", linewidth=2, linestyle='-')
plt.xlabel("Epochs")
plt.ylabel("Learning Rate")
plt.title("Comparison of Learning Rate Schedulers")
plt.legend(loc='best')
plt.grid(True)
plt.show()

# --- Individual subplots ---
plt.figure(figsize=(15, 10))

for idx, (lrs, name) in enumerate([
    (lrs_multistep, "MultiStepLR"),
    (lrs_exponential, "ExponentialLR"),
    (lrs_polynomial, "PolynomialLR"),
    (lrs_step, "StepLR"),
    (lrs_cosine, "CosineAnnealingLR"),
    (lrs_onecycle, "OneCycleLR (Warmup + Cosine)"),
], start=1):
    plt.subplot(3, 2, idx)
    plt.plot(lrs, label=name)
    plt.xlabel("Epochs")
    plt.ylabel("Learning Rate")
    plt.title(name)
    plt.grid(True)

plt.tight_layout()
plt.show()

### Interpreting the curves

- **StepLR** produces staircase drops -- the rate stays constant between milestones, then jumps down.
- **MultiStepLR** is similar but lets you place drops at non-uniform intervals (e.g., epochs 30, 60, 90).
- **ExponentialLR** decays smoothly but can reach very small values quickly ($\gamma^{100} \approx 2.66 \times 10^{-5}$ for $\gamma=0.9$).
- **PolynomialLR** with power 2 gives a concave decay: slow at first, then accelerating.
- **CosineAnnealingLR** follows a half-cosine from $\alpha_{\max}$ down to $\alpha_{\min}$, spending more time at medium rates -- this is often a sweet spot.
- **OneCycleLR** ramps up during warmup (first 30% of epochs), then decays via cosine annealing, finishing near zero.

## Training Experiment: CNN on Fashion-MNIST

We now test how these schedules perform in a real training loop. A small LeNet-style CNN is trained on Fashion-MNIST for 50 epochs with four optimizers (SGD, Adam, AdamW, SGD+Momentum) crossed with the six schedules above -- 24 runs in total. Training and test accuracy are recorded each epoch.

> **Requirements:** This cell uses [Weights & Biases](https://wandb.ai) (`wandb`) for experiment tracking. Install with `pip install wandb` and run `wandb login` before executing. If you prefer not to use wandb, you can remove the `wandb.*` calls and the results will still be plotted locally.

In [22]:
import torch
from torch import nn, optim
from torch.optim import lr_scheduler
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import wandb
import matplotlib.pyplot as plt

# Initialize Weights and Biases (wandb)
wandb.init(project="[teaching]-sds431-fall24-compare-learning-rates")

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define the neural network model
def net_fn():
    model = nn.Sequential(
        nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Conv2d(6, 16, kernel_size=5), nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Flatten(),
        nn.Linear(16 * 5 * 5, 120), nn.ReLU(),
        nn.Linear(120, 84), nn.ReLU(),
        nn.Linear(84, 10)
    )
    return model

# Load Fashion-MNIST dataset
batch_size = 256
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Define the training loop
def train(net, train_loader, test_loader, num_epochs, loss, optimizer, device, scheduler=None):
    net.to(device)
    train_losses, train_accuracies, test_accuracies = [], [], []
    for epoch in range(num_epochs):
        net.train()
        total_loss, correct, total = 0, 0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()

            total_loss += l.item() * X.size(0)
            _, predicted = torch.max(y_hat, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)

        train_loss = total_loss / total
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Evaluate on the test set
        net.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                y_hat = net(X)
                _, predicted = torch.max(y_hat, 1)
                correct += (predicted == y).sum().item()
                total += y.size(0)

        test_acc = correct / total
        test_accuracies.append(test_acc)

        # Log results to wandb
        wandb.log({"Epoch": epoch + 1, "Train Loss": train_loss, "Train Accuracy": train_acc,
                   "Test Accuracy": test_acc})

        # Scheduler step if defined
        if scheduler:
            scheduler.step()

        print(f'Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss:.4f}, '
              f'Train Accuracy: {train_acc:.4f}, Test Accuracy: {test_acc:.4f}')

    return train_losses, train_accuracies, test_accuracies

# Hyperparameters
num_epochs = 50  # Increase epochs to ensure convergence
learning_rate = 0.1

# Optimizers and Schedulers
optim_methods = {
    "SGD": optim.SGD,
    "Adam": optim.Adam,
    "AdamW": optim.AdamW,
    "Momentum": lambda params, lr: optim.SGD(params, lr=lr, momentum=0.9)
}

schedulers = {
    "MultiStepLR": lambda opt: lr_scheduler.MultiStepLR(opt, milestones=[15, 30, 45], gamma=0.1),
    "ExponentialLR": lambda opt: lr_scheduler.ExponentialLR(opt, gamma=0.9),
    "PolynomialLR": lambda opt: lr_scheduler.PolynomialLR(opt, total_iters=num_epochs, power=2),
    "StepLR": lambda opt: lr_scheduler.StepLR(opt, step_size=20, gamma=0.5),
    "CosineAnnealingLR": lambda opt: lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs),
    "OneCycleLR": lambda opt: lr_scheduler.OneCycleLR(
        opt,
        max_lr=learning_rate,
        total_steps=num_epochs,
        pct_start=0.3,
        anneal_strategy='cos',
        div_factor=25.0,
        final_div_factor=10000.0
    )
}

# Initialize storage for plotting
results = {}

# Train and evaluate each optimizer with each scheduler
for opt_name, opt_fn in optim_methods.items():
    for sched_name, sched_fn in schedulers.items():
        print(f"\nTraining {opt_name} with {sched_name}")
        wandb.run.name = f"{opt_name}_{sched_name}"
        model = net_fn()
        optimizer = opt_fn(model.parameters(), lr=learning_rate)
        scheduler = sched_fn(optimizer)

        # Train the model
        train_losses, train_accuracies, test_accuracies = train(
            model, train_loader, test_loader, num_epochs, nn.CrossEntropyLoss(), optimizer, device, scheduler
        )

        # Store results
        results[f"{opt_name}_{sched_name}"] = {
            "train_losses": train_losses,
            "train_accuracies": train_accuracies,
            "test_accuracies": test_accuracies
        }

# Plotting the training and testing curves
plt.figure(figsize=(20, 10))
for key, result in results.items():
    plt.plot(result['test_accuracies'], label=f"{key} Test Acc", linestyle='--')
    plt.plot(result['train_accuracies'], label=f"{key} Train Acc")

plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Testing Accuracy Curves')
plt.legend(loc='best')
plt.grid(True)
plt.show()

# Finish the wandb run
wandb.finish()


wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: zhuoran-yang (zhuoran_reseach). Use `wandb login --relogin` to force relogin


Using device: cpu


100%|██████████| 26421880/26421880 [00:01<00:00, 19280807.62it/s]


Extracting ./data/FashionMNIST/raw/train-images-idx3-ubyte.gz to ./data/FashionMNIST/raw



100%|██████████| 29515/29515 [00:00<00:00, 309354.96it/s]


...

100%|██████████| 4422102/4422102 [00:00<00:00, 5593593.51it/s]


Extracting ./data/FashionMNIST/raw/t10k-images-idx3-ubyte.gz to ./data/FashionMNIST/raw



100%|██████████| 5148/5148 [00:00<00:00, 4012688.53it/s]


Extracting ./data/FashionMNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/FashionMNIST/raw


Testing SGD with learning rate 0.01
Epoch 1/10, Train Loss: 2.2449, Train Accuracy: 0.1494, Test Accuracy: 0.2246
Epoch 2/10, Train Loss: 1.3093, Train Accuracy: 0.5386, Test Accuracy: 0.5974
Epoch 3/10, Train Loss: 0.8186, Train Accuracy: 0.6921, Test Accuracy: 0.7175
Epoch 4/10, Train Loss: 0.7382, Train Accuracy: 0.7212, Test Accuracy: 0.7300
Epoch 5/10, Train Loss: 0.6941, Train Accuracy: 0.7375, Test Accuracy: 0.7008
Epoch 6/10, Train Loss: 0.6423, Train Accuracy: 0.7594, Test Accuracy: 0.7521
Epoch 7/10, Train Loss: 0.6257, Train Accuracy: 0.7662, Test Accuracy: 0.7418
Epoch 8/10, Train Loss: 0.6123, Train Accuracy: 0.7714, Test Accuracy: 0.7599
Epoch 9/10, Train Loss: 0.5990, Train Accuracy: 0.7782, Test Accuracy: 0.7549
Epoch 10/10, Train Loss: 0.5870, Train Accuracy: 0.7816, Test Accuracy: 0.7727

Testing SGD with learning rate 0.05
Epoch 1/10, Train Loss: 1.4132, Train Accuracy: 0.4723, 

KeyboardInterrupt: 

## Summary

**Key takeaways from this notebook:**

- **Learning rate scheduling** is essential for training deep networks effectively. A fixed rate either converges too slowly or oscillates near the optimum.
- **Step-based schedules** (StepLR, MultiStepLR) are simple and interpretable but require manual tuning of milestones.
- **Cosine annealing** ($\alpha_t = \alpha_{\min} + \tfrac{1}{2}(\alpha_{\max} - \alpha_{\min})(1 + \cos(\pi t/T))$) is a robust default that spends more time at moderate learning rates and has strong empirical performance across domains.
- **Warmup + cosine (OneCycleLR)** often yields the fastest convergence ("super-convergence") by ramping up the rate initially to escape sharp minima, then smoothly decaying.
- **Exponential and polynomial schedules** offer smooth monotone decay and are useful when you want fine-grained control over the decay shape.
- The best schedule depends on the optimizer, dataset, and training budget. When in doubt, **cosine annealing** or **OneCycleLR** are strong starting points.